# Heart Disease Prediction with Ridge & Lasso Logistic Regression

End-to-end walkthrough of the project. Every step calls into the reusable modules
under `src/`, so the notebook and the production pipeline stay in sync — nothing
here is a one-off that can drift from the code the Flask/Streamlit apps serve.

**Contents**
1. Problem definition
2. Dataset
3. Data understanding
4. Data preprocessing
5. Exploratory data analysis
6. Feature selection & engineering
7. Train-test split
8. Model building
9. Model evaluation
10. Saving the model
11. Deployment


In [ ]:
%matplotlib inline

import os
import sys
from pathlib import Path

# Make the project root importable and the working directory predictable.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src import config, data_loader, eda, evaluate, model_card, predict, preprocessing, train

pd.set_option("display.max_columns", 50)
print("Project root:", ROOT)

## 1. Problem definition

Predict whether a patient has heart disease (**binary: present / absent**) from 13
routine clinical measurements. The model is a *decision-support* tool for doctors
and clinicians: it returns a predicted class **and** a calibrated-ish risk
probability so the clinician can weigh it alongside the full clinical picture.

We deliberately use **regularised logistic regression** because it is
interpretable (coefficients are log-odds per standardised unit), cheap to run,
and robust on a small dataset:

* **Ridge (L2)** shrinks coefficients but keeps every feature.
* **Lasso (L1)** shrinks some coefficients exactly to zero — embedded feature selection.

> This is a research/decision-support artefact, **not** a medical device.

## 2. Dataset

UCI *Cleveland* Heart Disease dataset — 303 records, 13 clinical features + 1 target.
The loader downloads the official UCI archive on first use and reads the
`processed.cleveland.data` member (it uses `?` for missing values in `ca` and `thal`).

The original `num` column encodes severity 0-4; we binarise it to
**0 = no disease, 1 = disease present**.

In [ ]:
df_raw = data_loader.load_dataset(binarize=True, save_snapshot=True)
print(f"Shape: {df_raw.shape[0]} records x {df_raw.shape[1]} columns")
df_raw.head()

In [ ]:
list(df_raw.columns)

## 3. Data understanding

Records/features, dtypes, missing values (especially `ca` and `thal`), duplicates,
unique values per categorical column, statistical summary and skew review.

In [ ]:
print("Data types:")
print(df_raw[config.FEATURES + ["target"]].dtypes.to_string())
print(f"\nDuplicate rows: {df_raw.duplicated().sum()}")
print(f"\nTarget distribution:\n{df_raw['target'].value_counts().to_string()}")

In [ ]:
print("Missing values:")
display(preprocessing.report_missing(df_raw))

print("\nUnique values per categorical column:")
uniques = preprocessing.unique_value_report(df_raw)
display(uniques[["column", "n_unique", "values"]])

In [ ]:
print("Statistical summary (continuous features):")
display(eda.statistical_summary(df_raw))

print("\nSkew review (|skew| > 1 indicates a markedly asymmetric distribution):")
skew = df_raw[config.NUMERIC_FEATURES].skew().sort_values(key=lambda s: s.abs(), ascending=False)
display(skew.round(3).to_frame("skew"))

## 4. Data preprocessing

* remove duplicate rows,
* **imputation:** median for numeric features, mode (most frequent) for `num_major_vessels` / `thalassemia`,
* **outlier detection:** IQR and z-score on `cholesterol` and `resting_bp` (flagged, not deleted — genuine high-risk patients),
* **range validation** against plausible physiological bounds,
* **encoding:** one-hot for `chest_pain_type`, `thalassemia`, `st_slope`, `sex`, …
* **scaling:** standardisation, required so Ridge/Lasso penalise coefficients fairly.

All of this lives in a single scikit-learn pipeline so training and inference
preprocess identically.

In [ ]:
df = preprocessing.clean_dataframe(df_raw)
print(f"Rows after dedup: {len(df)}")

print("\nOutlier report (flag only):")
display(preprocessing.report_outliers(df))

print("\nRange validation:")
display(preprocessing.validate_ranges(df))

In [ ]:
pipeline_preview = preprocessing.build_preprocessing_pipeline()
transformed = pipeline_preview.fit_transform(df[config.FEATURES])
print(f"Raw features: {len(config.FEATURES)} -> transformed matrix: {transformed.shape}")
print("\nTransformed feature names:")
print(preprocessing.feature_names(pipeline_preview))

## 5. Exploratory data analysis

Histograms, class-balance count plot, categorical-vs-target bars, box plots
(spread & outliers), age-vs-max-heart-rate scatter and the full correlation heatmap.
Figures are written to `reports/figures/`.

In [ ]:
eda_results = eda.run_eda(df)
eda_results["figures"]

In [ ]:
for name in ["histograms", "class_balance", "categorical_vs_target", "boxplots", "age_vs_hr", "correlation_heatmap"]:
    print(f"--- {name} ---")
    display(Image(filename=eda_results["figures"][name]))

## 6. Feature selection & engineering

We start with all 13 clinical features, then:

* drop features that are highly correlated with each other (none here — the strongest pair is well below the 0.85 threshold),
* flag near-zero-variance / near-zero-signal features (e.g. `fasting_blood_sugar`),
* add engineered features: `age_band` (binned age) and `age_chol` (age x cholesterol interaction),
* let **Lasso's L1 penalty do embedded selection** — the regularisation path below shows how many coefficients survive as `C` grows.

In [ ]:
print("Highly-correlated feature pairs:", preprocessing.correlation_redundancy(df) or "none")
print("\nAbsolute correlation with the target:")
display(preprocessing.feature_target_correlation(df).round(3).to_frame("abs_corr"))

In [ ]:
X_all, y_all = preprocessing.split_features_target(df)
lasso_path = train.lasso_feature_path(X_all, y_all)
display(lasso_path[["C", "n_selected", "n_zeroed", "train_accuracy"]].head(13))
evaluate.plot_lasso_path(lasso_path)
display(Image(filename=str(config.FIGURES_DIR / "lasso_feature_path.png")))

## 7. Train-test split

Stratified 80/20 split so the disease-present/absent ratio is preserved in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=y_all,
)
print(f"Train: {X_train.shape[0]} rows, prevalence {y_train.mean():.3f}")
print(f"Test : {X_test.shape[0]} rows, prevalence {y_test.mean():.3f}")

## 8. Model building

Three variants on the **same** split, with `C` tuned by stratified 5-fold CV (ROC-AUC):

* `baseline` — plain logistic regression, no penalty,
* `ridge_l2` — L2 penalty,
* `lasso_l1` — L1 penalty.

In [ ]:
results = train.train_all_variants(X_train, y_train, X_test, y_test)
print("\nBest variant:", train.select_best_model(results))

## 9. Model evaluation

Accuracy, precision, recall, F1, ROC-AUC, confusion matrices and ROC curves.
In a screening context **recall** matters most — missing a diseased patient is
costlier than a false alarm — which is why we also inspect threshold sensitivity.

In [ ]:
comparison = evaluate.build_comparison_table(results)
display(evaluate.highlight_best(comparison))

In [ ]:
evaluate.plot_confusion_matrices(results)
evaluate.plot_roc_curves(results, y_test)
display(Image(filename=str(config.FIGURES_DIR / "confusion_matrices.png")))
display(Image(filename=str(config.FIGURES_DIR / "roc_curves.png")))

In [ ]:
best_variant = train.select_best_model(results)
print(f"Threshold sensitivity for '{best_variant}':")
display(evaluate.threshold_sensitivity(results[best_variant]["model"], X_test, y_test))

In [ ]:
print("Non-zero coefficients per variant (Lasso performs feature selection):")
for variant in train.VARIANTS:
    coefs = train.feature_selection_report(results[variant]["model"])
    print(f"  {variant:9s}: {coefs['selected'].sum():2d}/{len(coefs)} non-zero")

display(train.feature_selection_report(results[best_variant]["model"]).head(10))

## 10. Saving the model

The winning model (pipeline + classifier) is serialised with `joblib` in three ways:

* an immutable **versioned** artifact, e.g. `ridge_l2_v1.0.0_20260101T000000Z.joblib`,
* a stable `*_latest.joblib` pointer the deployment app reads,
* `*_metadata.json` plus a human-readable `MODEL_CARD.md` (metrics + expected inputs).

In [ ]:
# Persist every variant so the web app can contrast Ridge and Lasso in the Live Simulation.
artifacts = train.persist_all_variants(results, X_train, y_train)
card = train.persist_best_model(results[best_variant], X_train, y_train, paths=artifacts[best_variant])
print('Persisted variants:', sorted(artifacts))
card

In [ ]:
print(Path(config.MODELS_DIR, "MODEL_CARD.md").read_text(encoding="utf-8")[:1800])

## 11. Deployment

Load the saved artifact exactly the way the apps do and score a patient.

### Run the apps

```bash
# REST API + clinician form
python app/flask_app.py            # http://localhost:5000

# Streamlit clinician UI
streamlit run app/streamlit_app.py # http://localhost:8501
```

### REST example

```bash
curl -X POST http://localhost:5000/predict \
  -H "Content-Type: application/json" \
  -d '{"features": {"age": 57, "sex": 1, "chest_pain_type": 4, "resting_bp": 140,
        "cholesterol": 260, "fasting_blood_sugar": 0, "resting_ecg": 1,
        "max_heart_rate": 140, "exercise_angina": 1, "st_depression": 2.0,
        "st_slope": 2, "num_major_vessels": 2, "thalassemia": 7}}'
```

Ops endpoints: `/health`, `/metrics`, `/model-info`.

In [ ]:
model, info = predict.load_model()
print("Serving:", info["name"], "v" + info["version"])

patient = {
    "age": 57, "sex": 1, "chest_pain_type": 4, "resting_bp": 140,
    "cholesterol": 260, "fasting_blood_sugar": 0, "resting_ecg": 1,
    "max_heart_rate": 140, "exercise_angina": 1, "st_depression": 2.0,
    "st_slope": 2, "num_major_vessels": 2, "thalassemia": 7,
}
predict.predict_patient(patient, model=model)